# Demo: Agentic Real-Time Metadata Standardization

## Configurations

Before running this notebook, create a `.env` file in the project root with the following API keys:

```
OPENAI_API_KEY=your-openai-api-key          # Required — for LLM calls
CEDAR_API_KEY=your-cedar-api-key            # Required — for fetching CEDAR templates
BIOPORTAL_API_KEY=your-bioportal-api-key    # Required — for ontology term lookups

LANGSMITH_API_KEY=your-langsmith-api-key    # Optional — enables observability for LLM calls and agent runs
LANGSMITH_TRACING=true                      # Optional
LANGSMITH_PROJECT=my-project-name           # Optional
```

See [.env.example](../.env.example) for a template.

`DATA_ROOT` points to the root data directory. The evaluation functions expect the following directory structure underneath it:

```
DATA_ROOT/
├── schemas/
│   ├── atacseq.json              # JSON Schema for each assay type
│   ├── lcms.json
│   └── ...
├── atacseq/                      # One directory per assay type
│   ├── input/
│   │   ├── atacseq-<hash>.json   # Legacy metadata records (input)
│   │   └── ...
│   ├── gold/
│   │   ├── atacseq-<hash>.json   # Gold-standard reference outputs
│   │   └── ...
│   └── output/
│       └── <MODEL>/              # e.g., "gpt5mini"
│           ├── baseline/
│           │   ├── atacseq-<hash>.json   # Prompt-only LLM outputs
│           │   └── ...
│           └── experiment/
│               ├── atacseq-<hash>.json   # Tool-augmented agent outputs
│               └── ...
├── lcms/
│   ├── input/ ...
│   ├── gold/ ...
│   └── output/ ...
└── ...
```

Gold-standard and output files share the same filenames so that each output can be matched to its reference for evaluation.

In [ ]:
DATA_ROOT = "../data"

`MODEL` specifies which LLM model was used for the migration run (e.g., `"gpt5mini"`).

In [ ]:
MODEL = "gpt5mini"

`RUN_TYPE` selects which output set to evaluate — either `"baseline"` (prompt-only LLM) or `"experiment"` (tool-augmented agent). This determines which subdirectory under each assay is used when loading output files for comparison against the gold standard.

In [ ]:
RUN_TYPE = "experiment"

## Run Evaluations

Run the standardization agent on every assay type. The loop calls `run_experiment` once per assay, reading inputs from `DATA_ROOT/<assay>/input` and writing outputs to `DATA_ROOT/<assay>/output/<MODEL>/<RUN_TYPE>`.

In [ ]:
RUN_EVALUATION = False  # Change to "True" to run the evaluation. Be careful it costs money to run it!

In [ ]:
# To run a single assay from the command line instead of bulk evaluation:
# uv run python -m evaluation \
#     --input "data/atacseq/input" \
#     --target-schema "https://repo.metadatacenter.org/templates/dd5e8653-81cf-470b-b71b-15cab421bb84" \
#     --output "data/atacseq/output/gpt5mini/experiment" \
#     --model "gpt-5-mini" \
#     --concurrent 8 \
#     --experiment

if RUN_EVALUATION:
    from functools import partial
    from pathlib import Path

    from assays import ASSAY_SCHEMAS
    from evaluate import run_experiment

    if MODEL == "gpt41mini":
        gpt_model = "gpt-4.1-mini"
    elif MODEL == "gpt5mini":
        gpt_model = "gpt-5-mini"
    elif MODEL == "gpt5":
        gpt_model = "gpt-5"
    else:
        raise ValueError(f"Unknown model name: {MODEL}")

    if RUN_TYPE == "baseline":
        from baseline import build_baseline_workflow as build_workflow
        from baseline import build_user_prompt
    else:
        from experiment import build_experiment_workflow as build_workflow
        from experiment import build_user_prompt

    workflow_factory = partial(build_workflow, model=gpt_model)

    for assay, schema_iri in ASSAY_SCHEMAS.items():
        print(f"Running {RUN_TYPE} for {assay}...")
        run_experiment(
            template_iri=schema_iri,
            input_dir=Path(DATA_ROOT) / assay / "input",
            output_dir=Path(DATA_ROOT) / assay / "output" / MODEL / RUN_TYPE,
            workflow_factory=workflow_factory,
            user_prompt_builder=build_user_prompt,
            max_concurrency=8,
        )
else:
    print("Skipping evaluation runs.")

## Data Analysis

### 1. Per-Assay Accuracy Summary

`create_per_assay_accuracy_summary` iterates over all assay directories, calls `apply_metrics` on each gold-standard/output file pair within an assay, and averages the results to produce a mean accuracy per assay. The returned DataFrame contains three metric columns:

- **ontology_constrained_field_accuracy** — accuracy on fields whose values must come from a controlled ontology.
- **non_ontology_constrained_field_accuracy** — accuracy on free-text or non-ontology fields.
- **all_field_accuracy** — combined accuracy across all fields.

In [ ]:
from data_analysis import create_per_assay_accuracy_summary

per_assay_df = create_per_assay_accuracy_summary(DATA_ROOT, MODEL, RUN_TYPE)
per_assay_df

### 2. Overall Accuracy Summary

`create_overall_accuracy_summary` computes aggregate accuracy from raw counts rather than averaging ratios. It accumulates the number of correct and total fields across every gold-standard/output file pair from all assays, then divides once at the end to produce a single accuracy value. This avoids the bias that can arise from averaging per-file or per-assay ratios when sample sizes differ.

In [ ]:
from data_analysis import create_overall_accuracy_summary

overall_df = create_overall_accuracy_summary(DATA_ROOT, MODEL, RUN_TYPE)
overall_df

## Plots

In [ ]:
from plots import plot_grouped_bar_chart

`plot_grouped_bar_chart` produces a side-by-side bar chart comparing **baseline** vs **experiment** accuracy for a chosen metric across all assay types. Available metrics for the `metric` parameter:

- `ontology_constrained_field_accuracy`
- `non_ontology_constrained_field_accuracy`
- `all_field_accuracy`

Set `show_error_bars=True` to display min/max error bars on each bar, showing the range of accuracy values within each assay.

### Ontology-Constrained Field Accuracy 

In [ ]:
plot_grouped_bar_chart(
    DATA_ROOT,
    MODEL,
    metric="ontology_constrained_field_accuracy",
    title="Ontology-Constrained Field Accuracy per Assay Type",
)

In [ ]:
plot_grouped_bar_chart(
    DATA_ROOT,
    MODEL,
    metric="non_ontology_constrained_field_accuracy",
    title="Non-Ontology-Constrained Field Accuracy per Assay Type",
)

In [ ]:
plot_grouped_bar_chart(
    DATA_ROOT,
    MODEL,
    metric="all_field_accuracy",
    title="All Field Accuracy per Assay Type",
)

## Error Analysis

`create_error_report` aggregates field-level prediction errors into a summary report. It compares each output file against its gold standard, identifies every field where the predicted value differs from the expected value, and groups identical error patterns together. The returned DataFrame contains the following columns:

- **error_type** — the category of mismatch (e.g., wrong value, missing value).
- **field_type** — whether the field is ontology-constrained or non-ontology-constrained.
- **expected_value** — the field name and its gold-standard value.
- **predicted_value** — the field name and the value produced by the model.
- **frequency** — how many times this exact error pattern occurred across all samples.
- **assays** — which assay types exhibited this error.

In [ ]:
from data_analysis import create_error_report

error_report_df = create_error_report(DATA_ROOT, MODEL, RUN_TYPE)
error_report_df